In [26]:
from  abc import ABC, abstractmethod
class Compte(ABC):
    def __init__(self, solde, titulaire):
        self.__solde = solde
        self.titulaire = titulaire

    def consulter_solde(self):
        return self.__solde

    def deposer(self, montant):
        if montant > 0:
            self.__solde += montant
            print(f"Dépôt de {montant}€ effectué.")
        else:
            print("Montant invalide.")

    @abstractmethod
    def retirer(self, montant):
        pass

    def getSolde(self):
        return self.__solde

    def setSolde(self, nvSolde):
        self.__solde = nvSolde

In [27]:
from datetime import datetime
class Transaction:
    def __init__(self, type_t, montant):
        self.type = type_t
        self.montant = montant
        self.date = datetime.now()

    def toString(self):
        return f"{self.date.strftime('%d/%m/%Y %H:%M')} | {self.type} : {self.montant} €"


In [28]:
class CompteCourant(Compte):
    def __init__(self, titulaire, solde=0):
        super().__init__(solde, titulaire)
        self.historique = []

    def deposer(self, montant, type_t="Dépôt"):
        if montant > 0:
            nvSolde = self.getSolde() + montant
            self.setSolde(nvSolde)
            self.historique.append(Transaction(type_t, montant))
            print(f"Dépôt de {montant}€ validé ({type_t}).")
        else:
            print("Montant invalide.")

    def retirer(self, montant):
        if self.getSolde() >= montant:
            nvSolde = self.getSolde() - montant
            self.setSolde(nvSolde)
            self.historique.append(Transaction("Retrait", montant))
            print(f"Retrait de {montant}€ effectué avec succès.")
        else:
            print("Opération invalide : Fonds insuffisants.")

    def virement(self, destinataire, montant):
        if montant <= self.getSolde():
            self.retirer(montant)
            destinataire.deposer(montant)
            print(f"Virement de {montant}€ vers {destinataire.titulaire} effectué.")
        else:
            print("Fonds insuffisants pour le virement.")

    def afficher_historique(self):
        print(f"\n--- Historique des transactions ({self.titulaire}) ---")
        for t in self.historique:
            print(t.toString())

In [29]:
class CompteEpargne(Compte):
    def __init__(self, titulaire, solde_initial=0, taux=2.0):
        super().__init__(solde_initial, titulaire)
        self.taux_interet = taux

    def calculer_interets(self):
        interets = self.getSolde() * (self.taux_interet / 100)
        self.setSolde(self.getSolde() + interets)
        print(f"Intérêts calculés : +{interets}€ (Taux: {self.taux_interet}%)")

    def retirer(self, montant):
        if self.getSolde() - montant >= 10:
            self.setSolde(self.getSolde() - montant)
            print(f"Retrait de {montant}€ effectué sur le compte épargne.")
        else:
            print("Action refusée : Un compte épargne doit garder un minimum de 10€.")

In [30]:
class Client:
    def __init__(self, nom):
        self.nom = nom
        self.comptes = []

    def ajouter_compte(self, compte):
        self.comptes.append(compte)

    def afficher_bilan(self):
        print(f"\n=== Bilan de {self.nom} ===")
        total = 0
        for cpte in self.comptes:
            solde_actuel = cpte.consulter_solde()
            print(f"- {type(cpte).__name__} : {solde_actuel}€")
            total += solde_actuel
        print(f"SOLDE TOTAL : {total}€")


In [31]:
# --- Test du programme ---
moi = Client("Salma")
c_courant = CompteCourant("Salma", 1000)
c_epargne = CompteEpargne("Salma", 5000, 3.5)

moi.ajouter_compte(c_courant)
moi.ajouter_compte(c_epargne)

c_courant.deposer(500, "Prime de stage")
c_courant.retirer(200)
c_epargne.calculer_interets()
c_courant.virement(c_epargne, 300)

moi.afficher_bilan()
c_courant.afficher_historique()

Dépôt de 500€ validé (Prime de stage).
Retrait de 200€ effectué avec succès.
Intérêts calculés : +175.00000000000003€ (Taux: 3.5%)
Retrait de 300€ effectué avec succès.
Dépôt de 300€ effectué.
Virement de 300€ vers Salma effectué.

=== Bilan de Salma ===
- CompteCourant : 1000€
- CompteEpargne : 5475.0€
SOLDE TOTAL : 6475.0€

--- Historique des transactions (Salma) ---
15/03/2026 17:06 | Prime de stage : 500 €
15/03/2026 17:06 | Retrait : 200 €
15/03/2026 17:06 | Retrait : 300 €
